## 1. 데이터셋 준비 

### 실습 데이터 소개 (Spaceship Titanic
#### 배경 스토리
> 2912년, 우주선 타이타닉호가 항성간 이동 중 시공간 이상(anomaly) 에 충돌.
> 승객 절반이 다른 차원으로 순간이동(Transported) 되어 사라짐.
> 구조대의 AI로서, 어떤 승객이 순간이동 됐는지 예측하라.

#### 데이터 구조
- 타깃: Transported — True(순간이동 됨) / False(생존)
#### 주요 피처:

| 컬럼 | 설명 | 타입 | 비고 |
|---|---|---|---|
| `PassengerId` | 승객 고유 ID | 문자열 | `그룹번호_개인번호` 형식 (예: `0013_01`) |
| `HomePlanet` | 출발 행성 | 범주형 | Earth / Europa / Mars |
| `CryoSleep` | 냉동수면 여부 | 불리언 | 냉동수면 중 서비스 지출 = 0 |
| `Cabin` | 객실 번호 | 문자열 | `갑판/번호/측면` 형식 (예: `B/0/P`) |
| `Destination` | 목적지 행성 | 범주형 | TRAPPIST-1e / 55 Cancri e / PSO J318.5-22 |
| `Age` | 나이 | 수치형 | |
| `VIP` | VIP 서비스 여부 | 불리언 | |
| `RoomService` | 룸서비스 지출액 | 수치형 | 선내 서비스 지출 5개 중 하나 |
| `FoodCourt` | 푸드코트 지출액 | 수치형 | 선내 서비스 지출 5개 중 하나 |
| `ShoppingMall` | 쇼핑몰 지출액 | 수치형 | 선내 서비스 지출 5개 중 하나 |
| `Spa` | 스파 지출액 | 수치형 | 선내 서비스 지출 5개 중 하나 |
| `VRDeck` | VR 덱 지출액 | 수치형 | 선내 서비스 지출 5개 중 하나 |
| `Name` | 승객 이름 | 문자열 | 성(family name)으로 그룹 추정 가능 |
| `Transported` | 순간이동 여부 | 불리언 | **타깃 변수** (True / False) |

## 2. 데이터셋 불러오기

In [11]:
import os
import zipfile
import kaggle
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parents[1]
DATA_DIR = PROJECT_ROOT / "data" / "spaceship-titanic"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("프로젝트 루트:", PROJECT_ROOT)   
print("저장 경로   :", DATA_DIR)        

# 다운로드
kaggle.api.competition_download_files(
    "spaceship-titanic",
    path=str(DATA_DIR)
)

# 압축 해제
with zipfile.ZipFile(DATA_DIR / "spaceship-titanic.zip", "r") as z:
    z.extractall(DATA_DIR)

print(os.listdir(DATA_DIR))

프로젝트 루트: c:\ml_note
저장 경로   : c:\ml_note\data\spaceship-titanic
['sample_submission.csv', 'spaceship-titanic.zip', 'test.csv', 'train.csv']


In [24]:
#### 데이터 셋 확인 
import pandas as pd
from pathlib import Path


data = pd.read_csv(DATA_DIR / "train.csv")
test_data  = pd.read_csv(DATA_DIR / "test.csv")

print("data shape:", data.shape)
print("test shape :", test_data.shape)
data.head()

data shape: (8693, 14)
test shape : (4277, 13)


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


## 3. EDA - 데이터 분석

### 기본정보 탐색 
#### 데이터 정보 
- 수치형 : 6
- 문자형 : 5
- 범주형 : 2

#### 타깃 비율 
- True : 50.4%
- False : 49.6%

In [25]:
# 기본 정보
print("-"*10," 기본 정보 " ,"-"*10)
print(data.info())

# 결측값 확인
print("-"*10," 결측값 " ,"-"*10)
print(data.isnull().sum())

----------  기본 정보  ----------
<class 'pandas.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   str    
 1   HomePlanet    8492 non-null   str    
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   str    
 4   Destination   8511 non-null   str    
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   str    
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(2), str(5)
memory usage: 891.5+ KB
None
----------  결측값  ----------
PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
De

In [26]:
# 타깃 클래스 비율 확인 
print("-"*10," 타깃 비율 " ,"-"*10)
print(data["Transported"].value_counts())
print(data["Transported"].value_counts(normalize=True).round(3))

# 수치형 피처 기초 통계
print("-"*10," 수치형 피처 기초 통계 " ,"-"*10)
data[["Age","RoomService","FoodCourt","ShoppingMall","Spa","VRDeck"]].describe().round(2)

----------  타깃 비율  ----------
Transported
True     4378
False    4315
Name: count, dtype: int64
Transported
True     0.504
False    0.496
Name: proportion, dtype: float64
----------  수치형 피처 기초 통계  ----------


,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
count,8514.00,8512.00,8510.00,8485.00,8510.00,8505.00
mean,28.83,224.69,458.08,173.73,311.14,304.85
std,14.49,666.72,1611.49,604.70,1136.71,1145.72
min,0.00,0.00,0.00,0.00,0.00,0.00
25%,19.00,0.00,0.00,0.00,0.00,0.00
50%,27.00,0.00,0.00,0.00,0.00,0.00
75%,38.00,47.00,76.00,27.00,59.00,46.00
max,79.00,14327.00,29813.00,23492.00,22408.00,24133.00


## 4.데이터 전처리 

In [31]:
# 타깃 분리
y = train["Transported"].astype(int)
train = train.drop(columns=["Transported"])
print("타깃 분리 완료 | y shape:", y.shape)

타깃 분리 완료 | y shape: (8693,)
